In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import VectorAssembler
import pandas as pd
from sklearn.datasets import fetch_california_housing

spark = SparkSession.builder \
    .appName('California_Housing_RandomForest') \
    .master('local[*]') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/15 20:28:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/15 20:28:56 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 58826)
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/socketserver.py", line 318, in _handle_request_noblock
    self.process_request(request, client_address)
    ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.13/socketserver.py", line 349, in process_request
    self.finish_request(request, client_address)
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda

In [3]:
housing = fetch_california_housing()
df_pandas = pd.DataFrame(housing.data, columns=housing.feature_names)
df_pandas['MedHouseVal'] = housing.target

df_spark = spark.createDataFrame(df_pandas)
df_clean = df_spark.distinct()

In [4]:
df_features = df_clean.withColumn(
    "RoomsPerHousehold", col('AveRooms') / col('AveBedrms')
).withColumn(
    "PopulationPerHousehold", col('Population') / col("AveOccup")
)

feature_cols = [
    "MedInc", "HouseAge", "AveRooms", "AveBedrms", "Population", 
    "AveOccup", "Latitude", "Longitude", "RoomsPerHousehold", "PopulationPerHousehold"
]
target_col = "MedHouseVal"

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_assembled = assembler.transform(df_features)

In [5]:
train_df, test_df = df_assembled.randomSplit([0.8, 0.2], seed=42)

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol=target_col,
    numTrees=90,                
    maxDepth=9,                 
    maxBins=64,                 
    featureSubsetStrategy="auto" 
)

rf_model = rf.fit(train_df)

predictions = rf_model.transform(test_df)

26/06/15 20:30:41 WARN DAGScheduler: Broadcasting large task binary with size 1835.1 KiB
26/06/15 20:30:42 WARN DAGScheduler: Broadcasting large task binary with size 3.4 MiB
26/06/15 20:30:43 WARN DAGScheduler: Broadcasting large task binary with size 6.3 MiB
26/06/15 20:30:44 WARN DAGScheduler: Broadcasting large task binary with size 1726.6 KiB
                                                                                

In [6]:
evaluator_rmse = RegressionEvaluator(labelCol=target_col, predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol=target_col, predictionCol="prediction", metricName="r2")

rmse = evaluator_rmse.evaluate(predictions)
r2 = evaluator_r2.evaluate(predictions)

print(f"Random Forest RMSE: {rmse:.4f}")
print(f"Random Forest R2 Score: {r2:.4f}")

Random Forest RMSE: 0.5563
Random Forest R2 Score: 0.7672
